# Accuracy Lab: Foundation Tuning
This notebook is for high-precision experimentation with model outputs. Use this to find the "Golden Configuration" for your extraction prompts.

### Accuracy Levers available here:
1. **Decoding Parameters:** Adjust `temperature` and `top_p`.
2. **Few-Shot Examples:** Provide the model with perfect examples of input -> output.
3. **Context Engineering:** Strip useless noise from OCR text before sending to LLM.

In [ ]:
import time
import json
import re
from openai import OpenAI
from paddleocr import PaddleOCRVL

# Assumes vLLM servers are running on 8000 and 8001
vlm = PaddleOCRVL(vl_rec_backend="vllm-server", vl_rec_server_url="http://localhost:8000/v1", vl_rec_api_model_name="PaddlePaddle/PaddleOCR-VL")
llm_client = OpenAI(base_url="http://localhost:8001/v1", api_key="EMPTY")
LLM_MODEL = "Qwen/Qwen3-4B-AWQ" # Update if you use a different model
print("Engines Initialized.")

## Step 1: Vision Extraction (The Raw Input)
Change the `IMAGE_PATH` to any document you want to test.

In [ ]:
IMAGE_PATH = "../invoices/invoice007.jpg"

res = vlm.predict(IMAGE_PATH)

In [ ]:
res

In [ ]:
def extract_and_combine_content(data):
    """Helper to extract content from PaddleOCRVL results."""
    combined_content = []
    if isinstance(data, list) and data:
        if 'parsing_res_list' in data[0] and isinstance(data[0]['parsing_res_list'], list):
            for item in data[0]['parsing_res_list']:
                content = None
                if hasattr(item, 'content'):
                    content = item.content
                elif isinstance(item, dict):
                    content = item.get('content')
                
                if content is not None:
                    combined_content.append(content)
    return '\n'.join(combined_content)


ocr_text = extract_and_combine_content(res)
print("--- RAW OCR TEXT ---")
print(ocr_text)
# print(ocr_text[:500] + "...")

## Step 2: Context Engineering (Noise Filtering)
Experiment with stripping out headers, footers, or repetitive text that might confuse the model.

In [ ]:
from bs4 import BeautifulSoup

def html_table_to_markdown(html_content):
    """Converts HTML <table> to Markdown table using BeautifulSoup."""
    soup = BeautifulSoup(html_content, 'html.parser')
    tables = soup.find_all('table')
    
    markdown_tables = []
    for table in tables:
        rows = table.find_all('tr')
        if not rows: continue
        
        md_rows = []
        for i, row in enumerate(rows):
            cols = row.find_all(['td', 'th'])
            cols_text = [c.get_text(strip=True) for c in cols]
            md_rows.append("| " + " | ".join(cols_text) + " |")
            
            # Add separator after header
            if i == 0:
                md_rows.append("| " + " | ".join(["---"] * len(cols)) + " |")
        
        markdown_tables.append("\n".join(md_rows))
    
    return "\n\n".join(markdown_tables) if markdown_tables else html_content



def clean_ocr_text(text):
    """Cleans OCR text: removes img tags and converts tables."""
    # 1. Remove <img ...> tags
    text = re.sub(r'<img[^>]*>', '', text)
    
    # 2. Extract <table> contents and convert to markdown
    def table_replacer(match):
        return html_table_to_markdown(match.group(0))
    
    cleaned_text = re.sub(r'<table>.*?</table>', table_replacer, text, flags=re.DOTALL)
    
    # 3. Clean up excessive whitespace
    cleaned_text = re.sub(r'\n\s*\n', '\n\n', cleaned_text)
    
    return cleaned_text.strip()

cleaned_text = clean_ocr_text(ocr_text)
print(cleaned_text)

## Step 3: Few-Shot Prompt Lab


In [ ]:

SYSTEM_PROMPT = """You are a precise data extraction assistant specialized in financial documents.
Extract information from the provided document image and return it strictly as a JSON object.
Use the OCR text extraction as a guide for text accuracy.

RULES:
1. DATES: All dates MUST be converted to YYYY-MM-DD format.
2. NUMBERS: Convert currency and quantities to float numbers (e.g., ,200.50 -> 1200.50).
3. NULLS: If a field is not present in the text, use null.
4. NESTING: Follow the exact nested structure provided below to separate Vendor vs Client details.
5. LINE ITEMS: Extract every row from tables into the line_items array.

TARGET JSON SCHEMA:
{
  "document_details": {
    "document_type": "string",
    "invoice_number": "string",
    "invoice_date": "YYYY-MM-DD",
    "due_date": "YYYY-MM-DD"
  },
  "vendor_details": {
    "company_name": "string",
    "person_name": "string",
    "address": "string",
    "contact_info": "string"
  },
  "client_details": {
    "company_name": "string",
    "person_name": "string",
    "address": "string",
    "contact_info": "string"
  },
  "line_items": [
    {
      "description": "string",
      "quantity": "float",
      "unit_price": "float",
      "line_total": "float"
    }
  ],
  "financials": {
    "subtotal": "float",
    "tax_amount": "float",
    "total_amount": "float"
  }
}
"""

## Google Gemma Vision Test (Single-Model Extraction)
Gemma is NOT SUPPORTED in NVDIA T4 architecture

In [ ]:
import base64
from openai import OpenAI

# NOTE: Gemma is listed as unsupported on T4 in current vLLM versions for vision.
# This cell is here for documentation and future testing.

## Qwen3-VL Vision Test (Single-Model Extraction)
Testing the latest **Qwen3-VL-4B-Instruct** using vLLM's Vision API with streaming and usage monitoring.

In [ ]:
import base64
from openai import OpenAI

# --- 1. Configuration ---
# NOTE: Run the vLLM server for Qwen3-VL on port 8003
QWEN3_VL_BASE_URL = "http://localhost:8003/v1"
QWEN3_VL_MODEL = "Qwen/Qwen3-VL-4B-Instruct"
qwen3_vl_client = OpenAI(base_url=QWEN3_VL_BASE_URL, api_key="EMPTY")

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# --- 2. Image & Prompt ---
TEST_IMAGE = "../invoices/108563_page-0001.jpg"
base64_image = encode_image(TEST_IMAGE)

print(f"Sending image {TEST_IMAGE} to Qwen3-VL Vision...")

# --- 3. Execution with Streaming ---
try:
    response = qwen3_vl_client.chat.completions.create(
        model=QWEN3_VL_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": SYSTEM_PROMPT},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}"
                        }
                    }
                ],
            }
        ],
        temperature=0.01,
        stream=True,
        stream_options={"include_usage": True}
    )

    print("\n--- QWEN3-VL VISION OUTPUT ---")
    token_usage = None
    
    for chunk in response:
        if chunk.choices and chunk.choices[0].delta.content is not None:
            print(chunk.choices[0].delta.content, end="", flush=True)
        if hasattr(chunk, 'usage') and chunk.usage is not None:
            token_usage = chunk.usage

    print("\n\n--- TOKEN CONSUMPTION ---")
    if token_usage:
        print(f"Prompt Tokens:     {token_usage.prompt_tokens}")
        print(f"Completion Tokens: {token_usage.completion_tokens}")
        print(f"Total Tokens:      {token_usage.total_tokens}")
        
except Exception as e:
    print(f"Error: {e}")